# 120 — PCP prototype: controlled post-generation Q-gradient pilot

This worker first benchmarks a frozen scalar MC/TD critic's action gradient against **recorded same-state candidate outcomes**. It then optionally evaluates new action chunks from the same restored simulator root: stock replay, positive Q-gradient, negative Q-gradient, and random controls matched in action RMS at 0.02 and 0.06 normalized policy units. Every control executes at most the first ten actions and shares the frozen P&P continuation and future seed. The historical stock and one fresh P&P chunk are also retained (nine candidates per root).

**This is a post-generation action-gradient prototype, not the original flow-step PCP corrector in `pnp/pcp.py`.** That corrector nudges a flow clean estimate and re-noises it; this pilot edits the completed stored proposal. Higher predicted Q alone does not establish better control. Current weak gradients make this exploratory. These 18 reused roots overlap the original training/validation cohorts; each record retains its source split. They are not an independent test set.

Use three copies with `SHARD_INDEX=0,1,2`. Start at `ROOT_LIMIT=1`; `None` expands to six roots per shard. The benchmark loads only cached context/candidate artifacts and the small critic, and runs on CPU without SmolVLA inference. Simulator execution is separately disabled by `RUN_INTERVENTION=False`.

Supply the checkpoint yourself by upload or Drive path. **The worker never looks for or uploads model checkpoints in Supabase.** It verifies the frozen snapshot digest, full architecture/state, and the preaction root-input contract. Missing or incompatible pilot roots fail clearly.

In [ ]:
SHARD_INDEX = 0  # 0, 1, or 2
ROOT_LIMIT = 1  # smoke default; None means all six roots in this shard
RUN_INTERVENTION = False
CHECKPOINT_PATH = ''  # /content/uploaded.pt or /content/drive/MyDrive/.../checkpoint.pt
UPLOAD_CHECKPOINT = False
MOUNT_DRIVE = False  # True for a Drive checkpoint
DEVICE = 'cpu'  # CPU is sufficient for the first benchmark


In [ ]:
# Install NVIDIA EGL only when simulator intervention is enabled, before MuJoCo imports.
if RUN_INTERVENTION:
    import re, subprocess
    driver = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
        text=True).splitlines()[0].strip()
    package = f"libnvidia-gl-{re.match(r'\d+', driver).group()}"
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    availability = subprocess.check_output(['apt-cache', 'policy', package], text=True)
    if 'Candidate: (none)' in availability:
        raise RuntimeError(f'{package} is unavailable in this runtime')
    subprocess.run(['apt-get', 'install', '-y', '--no-install-recommends', package], check=True)
    subprocess.run('ldconfig -p | grep libEGL_nvidia', shell=True, check=True)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if UPLOAD_CHECKPOINT:
    from google.colab import files
    from pathlib import Path
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one scalar critic .pt checkpoint.')
    CHECKPOINT_PATH = str(Path(next(iter(uploaded))).resolve())

EXTRAS = 'sim'
SETUP_ENV = RUN_INTERVENTION
REPO_REF = 'codex/smolvla-q-program'
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/codex/smolvla-q-program/pnp-vla/scripts/colab_bootstrap.py').read().decode())


In [ ]:
from pathlib import Path
from pnp.smolvla_pcp_pilot import run_pcp_pilot
assert CHECKPOINT_PATH and Path(CHECKPOINT_PATH).is_file(), 'Set CHECKPOINT_PATH or enable UPLOAD_CHECKPOINT.'
assert SHARD_INDEX in (0, 1, 2)
REPORT_PATH = f'/content/pcp_pilot_shard_{SHARD_INDEX}.json'
report = run_pcp_pilot(
    checkpoint_path=CHECKPOINT_PATH, shard_index=SHARD_INDEX,
    root_limit=ROOT_LIMIT, run_intervention=RUN_INTERVENTION,
    device=DEVICE, output_path=REPORT_PATH)
print({'report': REPORT_PATH, 'new_simulator_roots': report['new_roots']})


Read `gradient_projection_pair_accuracy` only on mixed-outcome roots, together with gradient RMS and score pair accuracy. Recorded candidate differences are finite comparisons, not true gradient labels. A zero/invalid gradient blocks intervention. The ±gradient/random probes use a fixed RMS radius and retain known preaction masks; no input uses candidate termination.

To run the simulator phase, enable `RUN_INTERVENTION`, choose a CUDA runtime, and rerun from setup. This writes full branch artifacts and checkpoint SHA/architecture/snapshot/split metadata to a new Supabase experiment; complete groups are skipped on restart. Compare each control against `stock_replay` at its radius and report paired root outcomes, including stock replay disagreement with historical stock. The one-root smoke and 18 overlapping roots do not support policy-performance claims.